# 04: Model evaluation & auditing

## Project: AgroVision
**Purpose:** Assess the generalization performance of the trained ResNet18 model using the hold-out **test set**. This notebook acts as the final quality gate before deployment.

### Key objectives:
1.  **Environment setup:** Configure paths and compute device (GPU/CPU).
2.  **Pipeline construction:** Reconstruct the Dataset class and apply deterministic transforms.
3.  **Model retrieval:** Load the best-performing weights (`best_model.pt`) into the architecture.
4.  **Inference:** Execute predictions on unseen data.
5.  **Auditing:** Generate the confusion matrix, classification report, and visualize failure modes.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

## 1. Environment & configuration
Define project paths and compute device.

In [ ]:
PROJECT_ROOT_DIR_PATH = Path.cwd().parent

DATA_DIR_PATH = PROJECT_ROOT_DIR_PATH / "data"
RAW_DATA_DIR_PATH = DATA_DIR_PATH / "raw"
PROCESSED_DATA_DIR_PATH = DATA_DIR_PATH / "processed"
DATASET_CLEAN_FILE_PATH = PROCESSED_DATA_DIR_PATH / "dataset_clean.csv"

ARTIFACTS_DIR_PATH = PROJECT_ROOT_DIR_PATH / "artifacts"
NORMALIZATION_STATS_FILE_PATH = ARTIFACTS_DIR_PATH / "normalization_stats.json"
MODELS_DIR_PATH = PROJECT_ROOT_DIR_PATH / "models"
MODELS_CHECKPOINTS_DIR_PATH = MODELS_DIR_PATH / "checkpoints"
BEST_MODEL_FILE_PATH = MODELS_CHECKPOINTS_DIR_PATH / "best_model.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")

## 2. Load Metadata & statistics
Import the processed dataset index and the normalization statistics computed during the EDA phase.

In [ ]:
df = pd.read_csv(DATASET_CLEAN_FILE_PATH)

classes = sorted(df["label"].unique())
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
idx_to_class = {i: cls_name for i, cls_name in enumerate(classes)}

with open(NORMALIZATION_STATS_FILE_PATH) as f:
    stats = json.load(f)
    mean = stats["mean"]
    std = stats["std"]

print(f"Classes: {classes}")
print(f"Normalization: mean={mean}, std={std}")

## 3. Custom dataset definition
Implement the `AgroVisionDataset` class to handle image loading and label encoding.

In [ ]:
class AgroVisionDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.root_dir / row["filepath"]

        image = Image.open(img_path).convert("RGB")
        label_str = row["label"]
        label = class_to_idx[label_str]

        if self.transform:
            image = self.transform(image)

        return image, label

## 4. Transforms & test loader
Prepare the inference pipeline using **deterministic transformations** (`Resize` + `CenterCrop`) to ensure reproducible evaluation.

In [ ]:
test_transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
)

test_df = df[df["split"] == "test"]

test_dataset = AgroVisionDataset(test_df, RAW_DATA_DIR_PATH, transform=test_transform)

BATCH_SIZE = 32
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

print(f"Test set size: {len(test_dataset)} images")

## 5. Model initialization
Initialize the ResNet18 architecture and load the best-performing weights saved during training.

In [ ]:
model = models.resnet18(weights=None)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(classes))

state_dict = torch.load(BEST_MODEL_FILE_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(state_dict)

model = model.to(DEVICE)
model.eval()

print(f"Model loaded from: {BEST_MODEL_FILE_PATH}")

## 6. Inference Loop
Iterate through the test DataLoader using `torch.no_grad()` to collect predictions and true labels.

In [ ]:
y_true = []
y_pred = []

print("Starting inference...")

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing"):
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

## 7. Quantitative analysis
Evaluate the model using the classification report (precision, recall, f1-score) and confusion matrix.

In [ ]:
print(classification_report(y_true, y_pred, target_names=classes))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion matrix: test set")
plt.show()

## 8. Qualitative analysis
Visualize specific misclassifications to identify failure modes (e.g., similar rot patterns or image quality issues).

In [ ]:
def denormalize(tensor, mean, std):
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return tensor * std + mean


y_true_arr = np.array(y_true)
y_pred_arr = np.array(y_pred)
error_indices = np.where(y_true_arr != y_pred_arr)[0]

print(f"Total misclassified images: {len(error_indices)}")

num_display = min(len(error_indices), 5)

if num_display > 0:
    fig, axes = plt.subplots(1, num_display, figsize=(15, 5))

    if num_display == 1:
        axes = [axes]

    for i, idx in enumerate(error_indices[:num_display]):
        img_tensor, _ = test_dataset[idx]
        img_denorm = denormalize(img_tensor, mean, std)
        img_display = transforms.ToPILImage()(img_denorm)

        true_lbl = idx_to_class[y_true_arr[idx]]
        pred_lbl = idx_to_class[y_pred_arr[idx]]

        axes[i].imshow(img_display)
        axes[i].set_title(f"True: {true_lbl}\nPred: {pred_lbl}")
        axes[i].axis("off")

    plt.tight_layout()

## 9. Summary & transition to production
The evaluation phase is complete. The model has been audited on the test set, providing a clear assessment of performance and potential failure modes.

**Key outcomes:**
1.  **Inference:** Validated predictions were generated for the unseen test set.
2.  **Metrics:** The confusion matrix and classification report were produced for the best-performing model artifacts.
3.  **Auditing:** Specific errors were visualized to identify semantic confusion or data quality issues.

**Next steps:**
1.  **Modularization:** The experimental "lab" phase (notebooks) is concluded. Development shifts to the production codebase within the `src/` directory.
2.  **Inference logic:** Preprocessing and model loading logic will be encapsulated into a dedicated service (e.g., `freshness_service.py`) to support the API layer.
3.  **API integration:** The trained model will be exposed through the `freshness_controller.py` to handle real-time image classification requests.